# GRPO Smoke Analysis

This notebook answers three practical questions about the Qwen2.5-0.5B GSM8K smoke runs:

1. Did the GRPO training pipeline produce checkpoints and usable completion logs?
2. Did the reward signal and prompt-leak penalty behave the way we intended?
3. Did the trained checkpoint beat the base model on the small evaluation harness?

This is not a paper-result notebook. It is a debugging and learning notebook for deciding whether the 0.5B GRPO path is trustworthy enough to move toward serving and quantization.

## How To Read This Notebook

The plots matter more than any single printed verdict. A good smoke run should show nonzero reward variation, parseable final answers, decreasing prompt leakage after the penalty, and no obvious evaluation regression against the base model.

Accuracy and PPL answer different questions. Accuracy checks whether generated final answers are right; reference PPL checks whether the model assigns higher likelihood to the official GSM8K solution under teacher forcing. Lower PPL is better, but it does not guarantee better generated-answer accuracy.

## Configuration

Edit these paths when analyzing a different run. The defaults point at the completed 300-step leak-penalty run and the earlier 100-step baseline smoke run on Great Lakes.

In [ ]:
from pathlib import Path
import ast
import json
import math
import re
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

RUN_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/ckpts/qwen2_5_0_5b_grpo_leak_penalty_300step_50gsm8k')
BASELINE_RUN_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/ckpts/smoke_qwen2_5_0_5b_grpo_100step')
FORMAT_RUN_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/ckpts/qwen2_5_0_5b_grpo_format_fix_100step')
LOG_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/repos/posttrain-quant-serve/logs')
JOB_ID = '51089453'

EVAL_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_train10_leak300')
BASELINE_EVAL_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_train10_100step')
FORMAT_EVAL_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_train10_format100')
NUM_GENERATIONS = 4

HELDOUT_EVALS = [
    {
        'label': 'test50_100step',
        'eval_dir': Path('/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_test50_100step'),
        'trained_model': BASELINE_RUN_DIR,
        'split': 'test',
        'limit': 50,
    },
    {
        'label': 'test50_leak300',
        'eval_dir': Path('/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_test50_leak300'),
        'trained_model': RUN_DIR,
        'split': 'test',
        'limit': 50,
    },
]

# This works whether Jupyter starts in the repo root or in notebooks/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists() and (REPO_ROOT.parent / 'scripts').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from gsm8k_reward import (
    extract_model_answer,
    extract_reference_answer,
    gsm8k_exact_match_reward,
    has_prompt_leak_after_answer,
)

plt.rcParams.update({
    'figure.figsize': (8, 4),
    'axes.grid': True,
    'grid.alpha': 0.25,
})

for name, path in {
    'RUN_DIR': RUN_DIR,
    'BASELINE_RUN_DIR': BASELINE_RUN_DIR,
    'FORMAT_RUN_DIR': FORMAT_RUN_DIR,
    'LOG_DIR': LOG_DIR,
    'EVAL_DIR': EVAL_DIR,
    'BASELINE_EVAL_DIR': BASELINE_EVAL_DIR,
    'FORMAT_EVAL_DIR': FORMAT_EVAL_DIR,
    'REPO_ROOT': REPO_ROOT,
}.items():
    print(f'{name}: {path}  exists={path.exists()}')

## Helper Functions

This cell hides the loading and column-detection plumbing. The later analysis cells should read like analysis: load artifacts, compute metrics, plot the trend, and state what is missing.

In [ ]:
def safe_display(df: pd.DataFrame, message: str = 'No rows to display.'):
    if df is None or df.empty:
        print(message)
    else:
        display(df)


def file_size_mib(path: Path) -> float:
    return path.stat().st_size / 1024**2


def pick_column(df: pd.DataFrame, exact=(), contains=()) -> str | None:
    if df is None or df.empty:
        return None
    lower_to_col = {str(c).lower(): c for c in df.columns}
    for name in exact:
        if name.lower() in lower_to_col:
            return lower_to_col[name.lower()]
    for col in df.columns:
        low = str(col).lower()
        if any(key in low for key in contains):
            return col
    return None


def read_completion_file(path: Path) -> pd.DataFrame | None:
    suffix = path.suffix.lower()
    try:
        if suffix == '.jsonl':
            rows = []
            for line in path.read_text(errors='replace').splitlines():
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
            return pd.DataFrame(rows)
        if suffix == '.json':
            obj = json.loads(path.read_text(errors='replace'))
            if isinstance(obj, list):
                return pd.DataFrame(obj)
            if isinstance(obj, dict):
                for value in obj.values():
                    if isinstance(value, list) and (not value or isinstance(value[0], dict)):
                        return pd.DataFrame(value)
                try:
                    return pd.DataFrame(obj)
                except ValueError:
                    return pd.DataFrame([obj])
        if suffix == '.csv':
            return pd.read_csv(path)
        if suffix == '.tsv':
            return pd.read_csv(path, sep='\t')
        if suffix == '.parquet':
            return pd.read_parquet(path)
    except Exception as exc:
        print(f'Could not read {path}: {exc!r}')
    return None


def infer_step_from_path(path: Path) -> int | None:
    match = re.search(r'(?:step|checkpoint|global_step|_)(\d+)', path.stem)
    return int(match.group(1)) if match else None


def load_completion_table(run_dir: Path) -> pd.DataFrame:
    completion_dir = run_dir / 'completions'
    if not completion_dir.exists():
        print(f'Missing completions directory: {completion_dir}')
        return pd.DataFrame()

    frames = []
    for path in sorted(p for p in completion_dir.rglob('*') if p.is_file()):
        df = read_completion_file(path)
        if df is None or df.empty:
            continue
        df = df.copy()
        df['source_file'] = str(path)
        if 'step' not in df.columns:
            inferred = infer_step_from_path(path)
            if inferred is not None:
                df['step'] = inferred
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def completion_columns(df: pd.DataFrame) -> dict[str, object]:
    return {
        'completion': pick_column(df, exact=('completion', 'completions', 'response', 'output', 'text'), contains=('completion', 'response', 'output')),
        'prompt': pick_column(df, exact=('prompt',), contains=('prompt',)),
        'answer': pick_column(df, exact=('answer', 'reference_answer', 'target'), contains=('reference', 'target', 'answer')),
        'advantage': pick_column(df, exact=('advantage',), contains=('advantage',)),
        'rewards': [c for c in df.columns if 'reward' in str(c).lower()],
    }


def analyze_completions(df: pd.DataFrame, label: str) -> tuple[pd.DataFrame, dict[str, object]]:
    cols = completion_columns(df)
    analysis = df.copy()
    metric = {'run': label, 'rows': len(analysis), 'completion_col_found': cols['completion'] is not None}

    if analysis.empty or cols['completion'] is None:
        return analysis, metric

    completion_col = cols['completion']
    analysis['_completion_text'] = analysis[completion_col].astype(str)
    analysis['parsed_answer'] = analysis['_completion_text'].map(extract_model_answer)
    analysis['has_extracted_answer'] = analysis['parsed_answer'].notna()
    analysis['prompt_leak'] = analysis['_completion_text'].map(has_prompt_leak_after_answer)
    analysis['completion_chars'] = analysis['_completion_text'].str.len()
    analysis['uses_hash_answer_format'] = analysis['_completion_text'].str.contains(r'####\s*[-+]?\d|####\s*<answer>', case=False, regex=True, na=False)
    analysis['uses_final_phrase'] = analysis['_completion_text'].str.contains(r'final\s+(?:numeric\s+)?answer|the\s+answer\s+is|answer\s+is', case=False, regex=True, na=False)

    metric.update({
        'answer_extraction_rate': float(analysis['has_extracted_answer'].mean()),
        'prompt_leak_rate': float(analysis['prompt_leak'].mean()),
        'completion_chars_mean': float(analysis['completion_chars'].mean()),
        'completion_chars_p95': float(analysis['completion_chars'].quantile(0.95)),
        'hash_format_rate': float(analysis['uses_hash_answer_format'].mean()),
        'final_phrase_rate': float(analysis['uses_final_phrase'].mean()),
    })

    if cols['answer'] is not None:
        answer_col = cols['answer']
        analysis['target_answer'] = analysis[answer_col].astype(str).map(extract_reference_answer)
        analysis['parsed_exact_match_reward'] = (
            analysis['parsed_answer'].notna()
            & analysis['target_answer'].notna()
            & (analysis['parsed_answer'] == analysis['target_answer'])
        ).astype(float)
        metric['parsed_reward_mean'] = float(analysis['parsed_exact_match_reward'].mean())
        metric['parsed_reward_max'] = float(analysis['parsed_exact_match_reward'].max())
        metric['parsed_reward_std'] = float(analysis['parsed_exact_match_reward'].std())

    for col in cols['rewards']:
        numeric = pd.to_numeric(analysis[col], errors='coerce')
        if numeric.notna().any():
            analysis[col] = numeric
            metric[f'{col}_mean'] = float(numeric.mean())
            metric[f'{col}_max'] = float(numeric.max())
            metric[f'{col}_std'] = float(numeric.std())

    return analysis, metric


def checkpoint_steps(run_dir: Path) -> list[int]:
    steps = []
    if not run_dir.exists():
        return steps
    for path in sorted(run_dir.glob('checkpoint-*')):
        match = re.search(r'checkpoint-(\d+)$', path.name)
        if match:
            steps.append(int(match.group(1)))
    return steps


def find_log_files(log_dir: Path, job_id: str):
    if not log_dir.exists():
        return [], []
    if job_id:
        out_files = sorted(log_dir.glob(f'*-{job_id}.out'))
        err_files = sorted(log_dir.glob(f'*-{job_id}.err'))
    else:
        out_files = sorted(log_dir.glob('*.out'))[-3:]
        err_files = sorted(log_dir.glob('*.err'))[-3:]
    return out_files, err_files


def parse_train_summaries(out_files: list[Path]) -> pd.DataFrame:
    rows = []
    for path in out_files:
        for line in path.read_text(errors='replace').splitlines():
            if line.startswith("{'train_runtime'"):
                try:
                    rows.append({'file': path.name, **ast.literal_eval(line)})
                except Exception as exc:
                    rows.append({'file': path.name, 'parse_error': str(exc), 'raw': line})
    return pd.DataFrame(rows)


def latest_trainer_state_path(run_dir: Path) -> Path | None:
    paths = []
    if (run_dir / 'trainer_state.json').exists():
        paths.append(run_dir / 'trainer_state.json')
    paths.extend(sorted(run_dir.glob('checkpoint-*/trainer_state.json')))
    if not paths:
        return None

    def step_key(path: Path) -> int:
        match = re.search(r'checkpoint-(\d+)', str(path))
        return int(match.group(1)) if match else -1

    return max(paths, key=step_key)


def load_trainer_history(run_dir: Path) -> pd.DataFrame:
    state_path = latest_trainer_state_path(run_dir)
    if state_path is None:
        print(f'No trainer_state.json found under {run_dir}. Per-step loss/KL may be unavailable.')
        return pd.DataFrame()
    try:
        state = json.loads(state_path.read_text())
    except Exception as exc:
        print(f'Could not read trainer state {state_path}: {exc!r}')
        return pd.DataFrame()
    history = pd.DataFrame(state.get('log_history', []))
    print(f'Loaded trainer history from {state_path}')
    return history


def first_existing_series(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lower = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    return None


def columns_containing(df: pd.DataFrame, *parts: str) -> list[str]:
    found = []
    for col in df.columns:
        low = str(col).lower()
        if all(part.lower() in low for part in parts):
            found.append(col)
    return found



def first_numeric_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    if df is None or df.empty:
        return None
    exact = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in exact:
            return exact[candidate.lower()]
    for candidate in candidates:
        parts = [part for part in re.split(r'[_/\s]+', candidate.lower()) if part]
        for col in df.columns:
            low = str(col).lower()
            if all(part in low for part in parts):
                series = pd.to_numeric(df[col], errors='coerce')
                if series.notna().any():
                    return col
    return None


def build_group_diagnostics(
    df: pd.DataFrame,
    reward_col: str | None,
    num_generations: int = 4,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if df is None or df.empty:
        print('No completion rows available for group diagnostics.')
        return pd.DataFrame(), pd.DataFrame()
    if reward_col is None or reward_col not in df.columns:
        print('No reward column available for group diagnostics.')
        return pd.DataFrame(), pd.DataFrame()
    if 'step' not in df.columns:
        print('No step column available for group diagnostics.')
        return pd.DataFrame(), pd.DataFrame()

    cols = completion_columns(df)
    work = df.copy()
    work['step'] = pd.to_numeric(work['step'], errors='coerce')
    work['_reward_numeric'] = pd.to_numeric(work[reward_col], errors='coerce')
    work = work.dropna(subset=['step', '_reward_numeric'])
    if work.empty:
        print('Reward/step columns were present but contained no numeric rows.')
        return pd.DataFrame(), pd.DataFrame()

    prompt_col = cols.get('prompt')
    if prompt_col and prompt_col in work.columns:
        group_cols = ['step', prompt_col]
        grouping_note = f'grouping by step + {prompt_col}'
    else:
        work['_synthetic_group'] = work.groupby('step').cumcount() // num_generations
        group_cols = ['step', '_synthetic_group']
        grouping_note = f'prompt column missing; grouping contiguous rows in chunks of {num_generations}'
    print('Group diagnostics:', grouping_note)

    advantage_col = cols.get('advantage')
    if advantage_col and advantage_col in work.columns:
        work['_advantage_numeric'] = pd.to_numeric(work[advantage_col], errors='coerce')
    else:
        work['_advantage_numeric'] = pd.NA

    rows = []
    for group_key, group in work.groupby(group_cols, dropna=False):
        rewards = pd.to_numeric(group['_reward_numeric'], errors='coerce').dropna()
        if rewards.empty:
            continue
        reward_std = float(rewards.std(ddof=0)) if len(rewards) > 1 else 0.0
        unique_rewards = int(rewards.nunique(dropna=True))
        logged_adv = pd.to_numeric(group['_advantage_numeric'], errors='coerce').dropna()
        if len(logged_adv):
            adv = logged_adv
            advantage_source = 'logged'
        elif reward_std > 0:
            adv = (rewards - rewards.mean()) / reward_std
            advantage_source = 'computed_from_reward'
        else:
            adv = pd.Series([0.0] * len(rewards))
            advantage_source = 'zero_reward_variance'
        rows.append({
            'step': float(group['step'].iloc[0]),
            'group_key': str(group_key),
            'num_completions': int(len(group)),
            'reward_mean': float(rewards.mean()),
            'reward_std': reward_std,
            'unique_rewards': unique_rewards,
            'zero_reward_variance': unique_rewards <= 1,
            'mean_advantage': float(adv.mean()) if len(adv) else 0.0,
            'mean_abs_advantage': float(adv.abs().mean()) if len(adv) else 0.0,
            'zero_advantage_group': bool((adv.abs() <= 1e-12).all()) if len(adv) else True,
            'advantage_source': advantage_source,
        })

    group_diag = pd.DataFrame(rows)
    if group_diag.empty:
        return group_diag, pd.DataFrame()
    step_diag = group_diag.groupby('step').agg(
        groups=('group_key', 'size'),
        reward_std_mean=('reward_std', 'mean'),
        reward_std_median=('reward_std', 'median'),
        frac_prompts_zero_reward_variance=('zero_reward_variance', 'mean'),
        mean_advantage=('mean_advantage', 'mean'),
        mean_abs_advantage=('mean_abs_advantage', 'mean'),
        frac_prompts_zero_advantage=('zero_advantage_group', 'mean'),
    ).reset_index().sort_values('step')
    return group_diag, step_diag

def load_eval_summary(eval_dir: Path) -> tuple[dict | None, pd.DataFrame]:
    summary_path = eval_dir / 'summary.json'
    paired_path = eval_dir / 'paired_comparison.csv'
    summary = None
    paired = pd.DataFrame()
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
    else:
        print(f'Missing eval summary: {summary_path}')
    if paired_path.exists():
        paired = pd.read_csv(paired_path)
    else:
        print(f'Missing paired comparison: {paired_path}')
    return summary, paired


def eval_sbatch_command(eval_dir: Path, trained_model: Path = RUN_DIR, limit: int = 10, split: str = 'train') -> str:
    return (
        'sbatch --account=cavestru0 --time=00:45:00 \\\n'
        f'  --export=ALL,EVAL_SPLIT={split},EVAL_LIMIT={limit},TRAINED_MODEL={trained_model},EVAL_OUTPUT_DIR={eval_dir} \\\n'
        '  slurm/eval_gsm8k_compare.sbatch'
    )

## 1. Artifact Checklist

This section checks whether the run created the files needed for analysis: model files, checkpoints, and completion logs. A checkpoint proves the training job reached a save point; completion logs prove we can inspect reward behavior instead of guessing from two printed examples.

In [ ]:
artifact_rows = []
if RUN_DIR.exists():
    for path in sorted(RUN_DIR.iterdir()):
        artifact_rows.append({
            'name': path.name,
            'kind': 'dir' if path.is_dir() else 'file',
            'size_mib': None if path.is_dir() else round(file_size_mib(path), 1),
        })
else:
    print(f'RUN_DIR does not exist: {RUN_DIR}')

safe_display(pd.DataFrame(artifact_rows))

steps = checkpoint_steps(RUN_DIR)
artifact_summary = pd.DataFrame([
    {'item': 'run_dir_exists', 'value': RUN_DIR.exists()},
    {'item': 'checkpoint_steps', 'value': steps},
    {'item': 'final_checkpoint_step', 'value': max(steps) if steps else None},
    {'item': 'completion_dir_exists', 'value': (RUN_DIR / 'completions').exists()},
])
display(artifact_summary)

## 2. Slurm And Trainer Log Summary

The Slurm logs tell us whether the cluster job completed and show coarse runtime. Trainer state may contain per-step metrics such as loss or KL, but TRL does not always save every series in the local artifacts. If a series is missing, the notebook says so directly instead of pretending it was measured.

In [ ]:
out_files, err_files = find_log_files(LOG_DIR, JOB_ID)
print('OUT files:')
for path in out_files:
    print(' ', path)
print('ERR files:')
for path in err_files:
    print(' ', path)

train_summary = parse_train_summaries(out_files)
safe_display(train_summary, 'No final train summary lines found in Slurm .out logs.')

for path in err_files:
    text = path.read_text(errors='replace')
    interesting = []
    for line in text.splitlines():
        low = line.lower()
        if any(key in low for key in ['warning', 'traceback', 'error', 'writing model shards', '100%', 'cancelled']):
            interesting.append(line[:240])
    print(f'\n=== selected stderr lines from {path.name} ===')
    if interesting:
        print('\n'.join(interesting[-40:]))
    else:
        print('No selected warnings/errors found.')

trainer_history = load_trainer_history(RUN_DIR)
if not trainer_history.empty:
    print('trainer_history columns:', list(trainer_history.columns))
    safe_display(trainer_history.tail(10))

## 3. Load Completion Artifacts

The completion table is the main evidence for the GRPO reward signal. Each row is one generated completion sampled during training. We can recompute answer parsing, prompt leakage, length, and reward summaries from these rows.

In [ ]:
completions = load_completion_table(RUN_DIR)
analysis, metrics = analyze_completions(completions, 'current_run')
metric_table = pd.DataFrame([{'metric': k, 'value': v} for k, v in metrics.items()])

print('completion rows:', len(completions))
print('columns:', list(completions.columns))
safe_display(completions.head(3), 'No completions loaded.')
safe_display(metric_table)

## 4. Training Dynamics

For GRPO, a useful training run should have nonconstant rewards: some completions should be better than others so the group-relative advantage has signal. A cleaner run should also reduce prompt-like leakage and avoid exploding completion length. KL and loss are plotted if the trainer saved them; if not, the absence is itself a logging gap to fix before scaling.

In [ ]:
reward_cols = [c for c in analysis.columns if 'reward' in str(c).lower()]
reward_col = reward_cols[0] if reward_cols else ('parsed_exact_match_reward' if 'parsed_exact_match_reward' in analysis.columns else None)

if analysis.empty or 'step' not in analysis.columns:
    print('No step column found in completions, so reward/leakage-over-step cannot be plotted.')
    grouped = pd.DataFrame()
    group_diag = pd.DataFrame()
    step_group_diag = pd.DataFrame()
else:
    dyn = analysis.copy()
    dyn['step'] = pd.to_numeric(dyn['step'], errors='coerce')
    dyn = dyn.dropna(subset=['step'])
    grouped = dyn.groupby('step').agg(
        rows=('step', 'size'),
        prompt_leak_rate=('prompt_leak', 'mean'),
        completion_chars_mean=('completion_chars', 'mean'),
        completion_chars_p95=('completion_chars', lambda s: s.quantile(0.95)),
    ).reset_index().sort_values('step')
    if reward_col is not None:
        grouped['reward_mean'] = dyn.groupby('step')[reward_col].mean().values
        grouped['completion_reward_std'] = dyn.groupby('step')[reward_col].std(ddof=0).fillna(0).values

    group_diag, step_group_diag = build_group_diagnostics(dyn, reward_col, NUM_GENERATIONS)
    if not step_group_diag.empty:
        grouped = grouped.merge(step_group_diag, on='step', how='left')

    safe_display(grouped.tail(12))

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()

    if 'reward_mean' in grouped.columns:
        axes[0].plot(grouped['step'], grouped['reward_mean'], marker='o')
        axes[0].set_title('Mean reward by logged step')
        axes[0].set_xlabel('step')
        axes[0].set_ylabel('reward')
    else:
        axes[0].text(0.5, 0.5, 'No reward column found', ha='center', va='center')
        axes[0].set_axis_off()

    reward_std_plot_col = 'reward_std_mean' if 'reward_std_mean' in grouped.columns else 'completion_reward_std'
    if reward_std_plot_col in grouped.columns:
        axes[1].plot(grouped['step'], grouped[reward_std_plot_col], marker='o', color='tab:green')
        axes[1].set_title('Within-group reward std by step')
        axes[1].set_xlabel('step')
        axes[1].set_ylabel('reward std')
    else:
        axes[1].text(0.5, 0.5, 'No reward std available', ha='center', va='center')
        axes[1].set_axis_off()

    if 'frac_prompts_zero_reward_variance' in grouped.columns:
        axes[2].plot(grouped['step'], grouped['frac_prompts_zero_reward_variance'], marker='o', color='tab:red')
        axes[2].set_title('Fraction of prompt groups with zero reward variance')
        axes[2].set_xlabel('step')
        axes[2].set_ylabel('fraction')
        axes[2].set_ylim(-0.05, 1.05)
    else:
        axes[2].text(0.5, 0.5, 'No group variance diagnostic', ha='center', va='center')
        axes[2].set_axis_off()

    if 'mean_abs_advantage' in grouped.columns:
        axes[3].plot(grouped['step'], grouped['mean_abs_advantage'], marker='o', color='tab:purple')
        axes[3].set_title('Mean absolute advantage by step')
        axes[3].set_xlabel('step')
        axes[3].set_ylabel('|advantage|')
    else:
        axes[3].text(0.5, 0.5, 'No advantage diagnostic', ha='center', va='center')
        axes[3].set_axis_off()

    if 'frac_prompts_zero_advantage' in grouped.columns:
        axes[4].plot(grouped['step'], grouped['frac_prompts_zero_advantage'], marker='o', color='tab:orange')
        axes[4].set_title('Fraction of prompt groups with zero advantage')
        axes[4].set_xlabel('step')
        axes[4].set_ylabel('fraction')
        axes[4].set_ylim(-0.05, 1.05)
    else:
        axes[4].text(0.5, 0.5, 'No zero-advantage diagnostic', ha='center', va='center')
        axes[4].set_axis_off()

    axes[5].plot(grouped['step'], grouped['completion_chars_mean'], marker='o', label='mean')
    axes[5].plot(grouped['step'], grouped['completion_chars_p95'], marker='o', label='p95')
    axes[5].set_title('Completion length by logged step')
    axes[5].set_xlabel('step')
    axes[5].set_ylabel('characters')
    axes[5].legend()

    fig.tight_layout()
    plt.show()

if trainer_history.empty:
    print('No trainer history available. Per-step KL/loss were not found in saved trainer_state.json artifacts.')
    print('New jobs keep logging_steps=1; if trainer_state.json is saved, this section will plot KL/reward_std/loss.')
else:
    step_col = first_existing_series(trainer_history, ['step', 'global_step'])
    x = trainer_history[step_col] if step_col else range(len(trainer_history))
    metric_groups = {
        'kl': ['kl'],
        'loss': ['loss'],
        'trainer_reward_std': ['reward_std', 'reward/std', 'rewards/std'],
        'frac_reward_zero_std': ['frac_reward_zero_std', 'frac_reward_zero/std', 'reward_zero_std'],
    }
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (name, candidates) in zip(axes.ravel(), metric_groups.items()):
        col = first_numeric_column(trainer_history, candidates)
        if col is None:
            ax.text(0.5, 0.5, f'No {name} series logged', ha='center', va='center')
            ax.set_axis_off()
            if name == 'kl':
                print('KL series not found in trainer_state.json. Old artifacts may not have saved it; rerun with current logging to check KL directly.')
            continue
        y = pd.to_numeric(trainer_history[col], errors='coerce')
        ax.plot(x, y, marker='o')
        ax.set_title(f'Trainer history: {name} ({col})')
        ax.set_xlabel(step_col or 'log row')
    fig.tight_layout()
    plt.show()

## 5. Reward Distribution

Answer extraction rate means the parser can find a number. Reward mean means the number is correct after applying any prompt-leak penalty. A run can have perfect extraction but still poor reward if the model confidently gives the wrong final answer.

In [ ]:
reward_summary_cols = ['answer_extraction_rate', 'prompt_leak_rate', 'completion_chars_mean', 'completion_chars_p95']
reward_summary = {k: metrics.get(k) for k in reward_summary_cols if k in metrics}
if reward_col is not None and reward_col in analysis.columns:
    rewards = pd.to_numeric(analysis[reward_col], errors='coerce').dropna()
    reward_summary.update({
        'reward_column': reward_col,
        'reward_mean': float(rewards.mean()) if len(rewards) else None,
        'reward_std': float(rewards.std()) if len(rewards) else None,
        'reward_min': float(rewards.min()) if len(rewards) else None,
        'reward_max': float(rewards.max()) if len(rewards) else None,
    })
    display(pd.DataFrame([reward_summary]).T.rename(columns={0: 'value'}))
    if len(rewards):
        ax = rewards.hist(bins=20)
        ax.set_title('Per-completion reward distribution')
        ax.set_xlabel('reward')
        ax.set_ylabel('completion count')
        plt.show()
else:
    print('No reward column found. The notebook can still inspect parsing and leakage, but not reward distribution.')
    display(pd.DataFrame([reward_summary]).T.rename(columns={0: 'value'}))

## 6. Leakage Before vs After The Penalty

This is the headline Day 2 question: did the explicit leak penalty reduce prompt-like text after the answer? The comparison recomputes leakage for both runs with the current detector, so it is an apples-to-apples comparison even if the detector changed after the original 100-step run.

In [ ]:
baseline_completions = load_completion_table(BASELINE_RUN_DIR)
baseline_analysis, baseline_metrics = analyze_completions(baseline_completions, 'baseline_100step')
current_analysis, current_metrics = analysis, metrics

comparison_metrics = pd.DataFrame([baseline_metrics, current_metrics])
safe_display(comparison_metrics)

if {'prompt_leak_rate', 'run'}.issubset(comparison_metrics.columns) and comparison_metrics['prompt_leak_rate'].notna().sum() >= 2:
    plot_df = comparison_metrics[['run', 'prompt_leak_rate']].dropna()
    ax = plot_df.plot(kind='bar', x='run', y='prompt_leak_rate', legend=False, color=['tab:gray', 'tab:blue'])
    ax.set_title('Prompt leak rate before vs after leak penalty')
    ax.set_ylabel('prompt leak rate')
    ax.set_xlabel('run')
    ax.set_ylim(0, max(0.05, plot_df['prompt_leak_rate'].max() * 1.25))
    for i, value in enumerate(plot_df['prompt_leak_rate']):
        ax.text(i, value, f'{value:.1%}', ha='center', va='bottom')
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()

    old = float(plot_df.iloc[0]['prompt_leak_rate'])
    new = float(plot_df.iloc[1]['prompt_leak_rate'])
    if new < old:
        print(f'Leakage improved: {old:.4f} -> {new:.4f} (delta {new - old:+.4f}).')
    elif new > old:
        print(f'Leakage worsened: {old:.4f} -> {new:.4f} (delta {new - old:+.4f}).')
    else:
        print(f'Leakage was flat at {new:.4f}.')
else:
    print('Need both baseline and current completion tables to plot leakage before/after.')

## 7. Base-vs-Trained Evaluation

Training reward is not enough. The real sanity check is running the base model and the trained checkpoint on the same GSM8K questions with the same decoding settings. This section loads `summary.json` and `paired_comparison.csv` if they exist; otherwise it prints the exact Slurm command to generate them.

In [ ]:
eval_summary, paired = load_eval_summary(EVAL_DIR)
baseline_eval_summary, baseline_paired = load_eval_summary(BASELINE_EVAL_DIR)
format_eval_summary, format_paired = load_eval_summary(FORMAT_EVAL_DIR)

def summary_row(label: str, summary: dict | None, status: str, path: Path) -> dict:
    if summary is None:
        return {'eval': label, 'status': status, 'path': str(path)}
    return {
        'eval': label,
        'status': status,
        'split': summary.get('split'),
        'limit': summary.get('limit'),
        'base_accuracy': summary.get('base', {}).get('accuracy'),
        'trained_accuracy': summary.get('trained', {}).get('accuracy'),
        'delta_accuracy': summary.get('delta_accuracy'),
        'base_prompt_leak_rate': summary.get('base', {}).get('prompt_leak_rate'),
        'trained_prompt_leak_rate': summary.get('trained', {}).get('prompt_leak_rate'),
        'base_reference_ppl': summary.get('base', {}).get('reference_ppl'),
        'trained_reference_ppl': summary.get('trained', {}).get('reference_ppl'),
        'reference_ppl_ratio': summary.get('reference_ppl_ratio'),
        'improved': summary.get('improved'),
        'worsened': summary.get('worsened'),
        'unchanged': summary.get('unchanged'),
        'path': str(path),
    }

if eval_summary is None:
    print('\nRun this on Great Lakes from the posttrain-quant-serve repo root to create the missing leak300 eval artifacts:')
    print(eval_sbatch_command(EVAL_DIR, RUN_DIR, limit=10, split='train'))
else:
    base = eval_summary.get('base', {})
    trained = eval_summary.get('trained', {})
    side_by_side = pd.DataFrame([
        {
            'model': 'base',
            'accuracy': base.get('accuracy'),
            'parse_rate': base.get('parse_rate'),
            'prompt_leak_rate': base.get('prompt_leak_rate'),
            'completion_chars_mean': base.get('completion_chars_mean'),
            'reference_ppl': base.get('reference_ppl'),
            'reference_nll_per_token': base.get('reference_nll_per_token'),
        },
        {
            'model': 'trained',
            'accuracy': trained.get('accuracy'),
            'parse_rate': trained.get('parse_rate'),
            'prompt_leak_rate': trained.get('prompt_leak_rate'),
            'completion_chars_mean': trained.get('completion_chars_mean'),
            'reference_ppl': trained.get('reference_ppl'),
            'reference_nll_per_token': trained.get('reference_nll_per_token'),
        },
    ])
    display(side_by_side)
    print('delta_accuracy:', eval_summary.get('delta_accuracy'))
    print('delta_reference_ppl:', eval_summary.get('delta_reference_ppl'))
    print('reference_ppl_ratio:', eval_summary.get('reference_ppl_ratio'))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, metric_name, title in [
        (axes[0], 'accuracy', 'Exact-match accuracy'),
        (axes[1], 'prompt_leak_rate', 'Eval prompt leak rate'),
        (axes[2], 'reference_ppl', 'Reference solution PPL'),
    ]:
        values = pd.to_numeric(side_by_side[metric_name], errors='coerce') if metric_name in side_by_side else pd.Series(dtype=float)
        if values.notna().any():
            ax.bar(side_by_side['model'], values, color=['tab:gray', 'tab:blue'])
            ax.set_title(title)
            ax.set_ylabel(metric_name)
            for i, value in enumerate(values):
                if pd.notna(value):
                    ax.text(i, value, f'{value:.3g}', ha='center', va='bottom')
        else:
            ax.text(0.5, 0.5, f'{metric_name} missing', ha='center', va='center')
            ax.set_axis_off()
    fig.tight_layout()
    plt.show()

    if not paired.empty:
        change_counts = paired['change'].value_counts().reindex(['improved', 'worsened', 'unchanged']).fillna(0)
        ax = change_counts.plot(kind='bar', color=['tab:green', 'tab:red', 'tab:gray'])
        ax.set_title('Paired question-level changes')
        ax.set_ylabel('question count')
        ax.set_xlabel('change')
        for i, value in enumerate(change_counts):
            ax.text(i, value, str(int(value)), ha='center', va='bottom')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
        display(paired)

comparison_rows = []
if baseline_eval_summary is not None:
    comparison_rows.append(summary_row('train10_100step', baseline_eval_summary, 'found', BASELINE_EVAL_DIR / 'summary.json'))
if eval_summary is not None:
    comparison_rows.append(summary_row('train10_leak300', eval_summary, 'found', EVAL_DIR / 'summary.json'))
if format_eval_summary is None:
    comparison_rows.append(summary_row('train10_format100', None, 'missing', FORMAT_EVAL_DIR / 'summary.json'))
    print('\nMissing format100 eval. Submit:')
    print(eval_sbatch_command(FORMAT_EVAL_DIR, FORMAT_RUN_DIR, limit=10, split='train'))
else:
    comparison_rows.append(summary_row('train10_format100', format_eval_summary, 'found', FORMAT_EVAL_DIR / 'summary.json'))

heldout_eval_rows = []
for spec in HELDOUT_EVALS:
    summary, _ = load_eval_summary(spec['eval_dir'])
    if summary is None:
        heldout_eval_rows.append(summary_row(spec['label'], None, 'missing', spec['eval_dir'] / 'summary.json'))
        print(f"\nMissing held-out eval {spec['label']}. Submit:")
        print(eval_sbatch_command(spec['eval_dir'], spec['trained_model'], limit=spec['limit'], split=spec['split']))
    else:
        heldout_eval_rows.append(summary_row(spec['label'], summary, 'found', spec['eval_dir'] / 'summary.json'))

all_eval_table = pd.DataFrame(comparison_rows + heldout_eval_rows)
safe_display(all_eval_table, 'No eval summaries found yet.')

train10_eval_table = pd.DataFrame(comparison_rows)
if not train10_eval_table.empty and {'eval', 'delta_accuracy', 'reference_ppl_ratio'}.issubset(train10_eval_table.columns):
    found_train10 = train10_eval_table[train10_eval_table['status'] == 'found'].copy()
    for col in ['delta_accuracy', 'base_prompt_leak_rate', 'trained_prompt_leak_rate', 'reference_ppl_ratio']:
        if col in found_train10:
            found_train10[col] = pd.to_numeric(found_train10[col], errors='coerce')
    if not found_train10.empty:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        plot_specs = [
            ('delta_accuracy', 'Train10 delta accuracy', 'trained - base'),
            ('trained_prompt_leak_rate', 'Train10 trained leak rate', 'leak rate'),
            ('reference_ppl_ratio', 'Train10 PPL ratio', 'trained / base'),
        ]
        for ax, col, title, ylabel in plot_specs:
            values = found_train10[col] if col in found_train10 else pd.Series(dtype=float)
            if values.notna().any():
                colors = ['tab:green' if v >= 0 else 'tab:red' for v in values.fillna(0)] if col == 'delta_accuracy' else 'tab:blue'
                ax.bar(found_train10['eval'], values, color=colors)
                ax.axhline(0 if col == 'delta_accuracy' else 1 if col == 'reference_ppl_ratio' else 0, color='black', linewidth=1)
                ax.set_title(title)
                ax.set_ylabel(ylabel)
                ax.tick_params(axis='x', rotation=20)
                for i, value in enumerate(values):
                    if pd.notna(value):
                        ax.text(i, value, f'{value:.3g}', ha='center', va='bottom')
            else:
                ax.text(0.5, 0.5, f'{col} missing', ha='center', va='center')
                ax.set_axis_off()
        fig.tight_layout()
        plt.show()

heldout_eval_table = pd.DataFrame(heldout_eval_rows)

if not heldout_eval_table.empty and 'delta_accuracy' in heldout_eval_table.columns:
    found = heldout_eval_table[heldout_eval_table['status'] == 'found'].copy()
    if not found.empty:
        ax = found.plot(kind='bar', x='eval', y='delta_accuracy', legend=False, color='tab:blue')
        ax.axhline(0, color='black', linewidth=1)
        ax.set_title('Held-out eval delta accuracy')
        ax.set_ylabel('trained - base accuracy')
        ax.set_xlabel('eval')
        plt.xticks(rotation=15, ha='right')
        plt.tight_layout()
        plt.show()

## 8. Inspect Concrete Examples

Aggregate metrics can hide failure modes. This section shows high-reward, low-reward, and leaky completions so you can see whether the reward and leakage detector agree with human judgment.

In [ ]:
if analysis.empty or completion_columns(analysis)['completion'] is None:
    print('No completion table available to inspect.')
else:
    cols = completion_columns(analysis)
    completion_col = cols['completion']
    show_cols = []
    for col in ['step', cols['prompt'], completion_col, 'parsed_answer', 'target_answer', reward_col, cols['advantage'], 'prompt_leak', 'completion_chars', 'source_file']:
        if col and col in analysis.columns and col not in show_cols:
            show_cols.append(col)

    if reward_col and reward_col in analysis.columns:
        print('Highest reward examples')
        display(analysis.sort_values(reward_col, ascending=False)[show_cols].head(5))
        print('Lowest reward examples')
        display(analysis.sort_values(reward_col, ascending=True)[show_cols].head(5))
    else:
        display(analysis[show_cols].head(10))

    if 'prompt_leak' in analysis.columns:
        print('Prompt-leak examples')
        safe_display(analysis[analysis['prompt_leak']][show_cols].head(5), 'No prompt-leak examples detected by the current detector.')

## 9. Parser Sandbox

Use this cell when a completion looks suspicious. It shows what number the parser extracts, whether leak detection fires after that answer, and what reward the current GSM8K reward function would assign.

In [ ]:
sandbox_cases = [
    {
        'case': 'clean_correct',
        'completion': 'We compute 6 * 52 = 312. Final answer: 312.',
        'reference': 'The answer is 312. #### 312',
    },
    {
        'case': 'leaky_correct',
        'completion': 'We compute 6 * 52 = 312. Final answer: 312.\n\nYou are given a list of integers. Write a program...',
        'reference': 'The answer is 312. #### 312',
    },
    {
        'case': 'unparseable',
        'completion': 'I cannot determine the final number from the information provided.',
        'reference': 'The answer is 312. #### 312',
    },
]

sandbox_rows = []
for item in sandbox_cases:
    reward = gsm8k_exact_match_reward([item['completion']], [item['reference']])[0]
    sandbox_rows.append({
        'case': item['case'],
        'parsed_model_answer': extract_model_answer(item['completion']),
        'parsed_reference_answer': extract_reference_answer(item['reference']),
        'prompt_leak_after_answer': has_prompt_leak_after_answer(item['completion']),
        'reward': reward,
        'completion': item['completion'],
    })
display(pd.DataFrame(sandbox_rows))

# Paste an arbitrary completion/reference here when debugging by hand.
sample_completion = 'Final answer: 312 pages per year.'
sample_reference = '... #### 624'
print('manual sample parsed model answer:', extract_model_answer(sample_completion))
print('manual sample parsed reference answer:', extract_reference_answer(sample_reference))
print('manual sample prompt leak:', has_prompt_leak_after_answer(sample_completion))
print('manual sample reward:', gsm8k_exact_match_reward([sample_completion], [sample_reference]))

## 10. Final Scorecard

The scorecard is intentionally compact. It should tell you whether the current run is a useful artifact, whether leakage improved, whether the trained checkpoint beat the base model, and what the next single action should be.

In [ ]:
def status_text(condition: bool, yes: str, no: str) -> str:
    return yes if condition else no

final_step = max(checkpoint_steps(RUN_DIR)) if checkpoint_steps(RUN_DIR) else None
reward_max = None
reward_mean = None
if reward_col and reward_col in analysis.columns:
    reward_values = pd.to_numeric(analysis[reward_col], errors='coerce').dropna()
    if len(reward_values):
        reward_max = float(reward_values.max())
        reward_mean = float(reward_values.mean())

leak_trend = 'unknown'
if 'comparison_metrics' in globals() and {'run', 'prompt_leak_rate'}.issubset(comparison_metrics.columns):
    leak_values = comparison_metrics.dropna(subset=['prompt_leak_rate'])
    if len(leak_values) >= 2:
        old = float(leak_values.iloc[0]['prompt_leak_rate'])
        new = float(leak_values.iloc[1]['prompt_leak_rate'])
        if new < old:
            leak_trend = f'improved ({old:.3f} -> {new:.3f})'
        elif new > old:
            leak_trend = f'worse ({old:.3f} -> {new:.3f})'
        else:
            leak_trend = f'flat ({new:.3f})'

acc_delta = None
ppl_ratio = None
train10_evidence = 'no train10 eval summary loaded'
if 'train10_eval_table' in globals() and isinstance(train10_eval_table, pd.DataFrame) and not train10_eval_table.empty:
    found_train10 = train10_eval_table[train10_eval_table['status'] == 'found'].copy()
    if not found_train10.empty:
        train10_evidence = '; '.join(
            f"{row['eval']}: delta={row.get('delta_accuracy')}, leak={row.get('trained_prompt_leak_rate')}, ppl_ratio={row.get('reference_ppl_ratio')}"
            for _, row in found_train10.iterrows()
        )
        preferred = found_train10[found_train10['eval'] == 'train10_format100']
        if preferred.empty:
            preferred = found_train10.tail(1)
        acc_delta = preferred.iloc[0].get('delta_accuracy')
        ppl_ratio = preferred.iloc[0].get('reference_ppl_ratio')
elif 'eval_summary' in globals() and eval_summary is not None:
    acc_delta = eval_summary.get('delta_accuracy')
    ppl_ratio = eval_summary.get('reference_ppl_ratio')
    train10_evidence = f'leak300: delta={acc_delta}, ppl_ratio={ppl_ratio}'

collapse_status = 'missing'
collapse_evidence = 'group diagnostics not available'
if 'grouped' in globals() and isinstance(grouped, pd.DataFrame) and 'frac_prompts_zero_reward_variance' in grouped.columns:
    recent = grouped.dropna(subset=['frac_prompts_zero_reward_variance']).tail(5)
    if len(recent):
        recent_frac = float(recent['frac_prompts_zero_reward_variance'].mean())
        collapse_status = 'possible' if recent_frac >= 0.5 else 'not obvious'
        collapse_evidence = f'recent zero-reward-variance group fraction={recent_frac:.3f}'

heldout_status = 'missing'
heldout_evidence = 'no held-out summaries loaded'
if 'heldout_eval_table' in globals() and isinstance(heldout_eval_table, pd.DataFrame) and not heldout_eval_table.empty:
    found = heldout_eval_table[heldout_eval_table['status'] == 'found']
    if len(found):
        heldout_status = 'available'
        heldout_evidence = '; '.join(
            f"{row['eval']}: delta={row.get('delta_accuracy')}" for _, row in found.iterrows()
        )
    else:
        heldout_status = 'pending'
        heldout_evidence = 'run the printed test50 sbatch commands'

if heldout_status in {'missing', 'pending'}:
    next_action = 'Run held-out test50 evals and inspect reward-variance diagnostics before another long GRPO run.'
elif collapse_status == 'possible':
    next_action = 'Try a gated diagnostic run with larger num_generations and Dr. GRPO/DAPO-style knobs, not more steps first.'
elif acc_delta is not None and acc_delta < 0:
    next_action = 'Inspect worsened paired examples before more training or quantization.'
else:
    next_action = 'Run a small held-out test eval before moving to serving/quantization.'

scorecard = pd.DataFrame([
    {'area': 'pipeline', 'status': status_text(final_step is not None, 'checkpoint found', 'missing checkpoint'), 'evidence': f'final_checkpoint_step={final_step}'},
    {'area': 'reward signal', 'status': status_text(reward_max is not None and reward_max > 0, 'nonzero reward exists', 'missing/dead reward'), 'evidence': f'mean={reward_mean}, max={reward_max}'},
    {'area': 'advantage/reward-std collapse', 'status': collapse_status, 'evidence': collapse_evidence},
    {'area': 'leakage trend', 'status': leak_trend, 'evidence': 'current detector recomputed on both runs'},
    {'area': 'train10 base-vs-trained accuracy', 'status': 'available' if acc_delta is not None else 'missing', 'evidence': train10_evidence},
    {'area': 'reference PPL', 'status': 'available' if ppl_ratio is not None else 'missing', 'evidence': f'preferred_or_latest_ppl_ratio={ppl_ratio}'},
    {'area': 'held-out evals', 'status': heldout_status, 'evidence': heldout_evidence},
    {'area': 'next action', 'status': next_action, 'evidence': ''},
])
display(scorecard)